In [ ]:
import numpy as np
import pickle
import matplotlib.pyplot as plt
import os
import seaborn as sns
import pickle
import torch
from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
from data.metrics import uceloss, sigma_scaling
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
from sklearn.preprocessing import MinMaxScaler
from omegaconf import OmegaConf
import yaml
import argparse

In [ ]:
# 24, 42
subject_indices = [1,13,26,27,29,34,35,41,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
explanation_function_names = ["IntegratedGradient", "GradientShap", "random_baseline", "Saliency"]
ks = (60*900)*np.arange(0.1, 1.1 ,0.1)
ks = ks.astype(int)
#save_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/ROAD/ROAD_subj1"
Reps = 10
acc_keys = ["rolling_binary_acc", "fixed_binary_acc"]

In [ ]:
grand_means_all_subjects= {}
for subject_index in subject_indices:

    save_path = f"evaluation/S4_S4EEGNet_ema_100_cal_py_{subject_index}__subject_{subject_index}"
    grand_mean = {k: {e: [] for e in explanation_function_names} for k in acc_keys}
    for k in ks:
        fig, axs = plt.subplots(nrows=2,ncols=2, figsize=(12,10))
        fig.suptitle(f"Subject {subject_index} k={k}")
        for explanaition_func_name in explanation_function_names:
            subject_data =  {k:[] for k in acc_keys}
            subject_data_mean =  {k:[] for k in acc_keys}
            subject_data_median =  {k:[] for k in acc_keys}
            for rep in range(Reps):

                file_path = os.path.join(save_path.format(subject_index), f"ROAD_all_metrics_subject_{subject_index}_rep_{rep}_explantion_func_{explanaition_func_name}_k_{k}.npz".format(rep, explanaition_func_name, k))
                data = np.load(file_path)
                for acc in acc_keys:
                    subject_data[acc].append(np.array([data[acc]]))

            for acc in acc_keys:
                mean_acc = np.mean(subject_data[acc],axis=0)
                median_acc = np.median(subject_data[acc],axis=0)
                subject_data_mean[acc].append(mean_acc.squeeze(0))
                subject_data_median[acc].append(median_acc.squeeze(0))
                grand_mean[acc][explanaition_func_name].append(np.mean(subject_data[acc]))
        
            
            for idx,acc in enumerate(acc_keys):
                if idx == 1:
                    axs[0,idx].plot(subject_data_mean[acc][0], label=explanaition_func_name)
                else:
                    axs[0,idx].plot(subject_data_mean[acc][0])
                #axs[0,idx].plot(subject_data_mean[acc][0])
                axs[1,idx].plot(subject_data_median[acc][0])
                axs[0,0].set_title("mean rolling binary acc")
                axs[0,1].set_title("mean fixed binary acc")
                axs[1,0].set_title("median rolling binary acc")
                axs[1,1].set_title("median fixed binary acc")
                axs[0,0].set_ylim([0,1])
                axs[0,1].set_ylim([0,1])
                axs[1,0].set_ylim([0,1])
                axs[1,1].set_ylim([0,1])
            fig.legend()
            fig.savefig(f"subject_index_{subject_index}_k_{k}.png")
    grand_means_all_subjects[subject_index] = grand_mean



In [ ]:
#    fig, axs = plt.subplots(nrows=2,ncols=1, figsize=(12,10))
#    for idx,acc in enumerate(acc_keys):
#        for explanaition_func_name in explanation_function_names:
#            if idx==0:
#                axs[idx].plot(ks, grand_mean[acc][explanaition_func_name], label=explanaition_func_name)
#            else:
#                axs[idx].plot(ks, grand_mean[acc][explanaition_func_name])
#    fig.legend()
#fig.savefig("grand_mean.png")

In [ ]:
for subject_index in [1,13,26,27,29,34,35,41,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]:
    fig, axs = plt.subplots(nrows=2,ncols=1, figsize=(12,10))
    for idx,acc in enumerate(acc_keys):
        for explanaition_func_name in explanation_function_names:
            if idx==0:
                axs[idx].plot(ks, grand_means_all_subjects[subject_index][acc][explanaition_func_name], label=explanaition_func_name)
            else:
                axs[idx].plot(ks, grand_means_all_subjects[subject_index][acc][explanaition_func_name])
    fig.legend()
    fig.savefig(f"grand_mean_subject_{subject_index}.png")

In [ ]:
subject_indices

In [ ]:
# Calculate average performance across all subjects for each explanation function
avg_performance = {acc_key: {func: [] for func in explanation_function_names} for acc_key in acc_keys}
std_performance = {acc_key: {func: [] for func in explanation_function_names} for acc_key in acc_keys}

for acc_key in acc_keys:
    for func in explanation_function_names:
        # For each k value, calculate the average and std across all subjects
        for k_idx in range(len(ks)):
            values = [grand_means_all_subjects[subject_idx][acc_key][func][k_idx] 
                     for subject_idx in subject_indices]
            avg_across_subjects = np.mean(values)
            std_across_subjects = np.std(values)
            
            avg_performance[acc_key][func].append(avg_across_subjects)
            std_performance[acc_key][func].append(std_across_subjects)

# Create percentage labels for the x-axis (10% to 100%)
percentages = np.arange(10, 101, 10)
percentage_labels = [f"{p}%" for p in percentages]

# Set up a nicer aesthetic style
fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(13, 8), sharex=True, sharey=True)
fig.subplots_adjust(hspace=0.3)
fig.suptitle("Average Performance Across All Subjects", fontsize=20, fontweight='bold')

# Use a better color palette
colors = plt.cm.viridis(np.linspace(0, 0.9, len(explanation_function_names)))

# Create a mapping for display names
display_names = {
    'random_baseline': 'Random Baseline',
    'Saliency': 'Input Gradient',
    'IntegratedGradient': 'IntegratedGradient',
    'GradientShap': 'GradientShap'
}

for idx, acc_key in enumerate(acc_keys):
    # Add horizontal grid lines
    axs[idx].yaxis.grid(True, linestyle='--', alpha=0.3, color='gray')
    
    for i, func in enumerate(explanation_function_names):
        color = colors[i]
        mean = np.array(avg_performance[acc_key][func])
        display_name = display_names.get(func, func)
        
        # Plot line with error bands
        axs[idx].plot(percentages, mean, label=display_name, linewidth=2, color=color)
    
    axs[idx].set_title(f"{acc_key.replace('_', ' ').title()}", fontsize=18, pad=10)
    axs[idx].set_xlabel("Percentage of imputed features", fontsize=16)
    axs[idx].set_ylabel("Accuracy", fontsize=16)
    axs[idx].set_ylim([0.45, 1])
    axs[idx].set_xticks(percentages)
    axs[idx].set_xticklabels(percentage_labels)
    
    # Improve tick labels
    axs[idx].tick_params(axis='both', which='major', labelsize=15)
    
    # Remove vertical grid
    axs[idx].xaxis.grid(False)
    
    # Add a border
    for spine in axs[idx].spines.values():
        spine.set_visible(True)
        spine.set_color('#ddd')
        spine.set_linewidth(0.8)

# Add a single legend at the bottom of the figure with horizontal layout
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=15, 
           bbox_to_anchor=(0.5, -0.03), frameon=True, framealpha=0.9)

plt.savefig("ROAD_average_performance_with_grid.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Calculate average performance across all subjects for each explanation function
avg_performance = {acc_key: {func: [] for func in explanation_function_names} for acc_key in acc_keys}
std_performance = {acc_key: {func: [] for func in explanation_function_names} for acc_key in acc_keys}

for acc_key in acc_keys:
    for func in explanation_function_names:
        # For each k value, calculate the average and std across all subjects
        for k_idx in range(len(ks)):
            values = [grand_means_all_subjects[subject_idx][acc_key][func][k_idx] 
                     for subject_idx in subject_indices]
            avg_across_subjects = np.mean(values)
            std_across_subjects = np.std(values)
            
            avg_performance[acc_key][func].append(avg_across_subjects)
            std_performance[acc_key][func].append(std_across_subjects)


fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(12, 10))
fig.suptitle("Average Performance Across All Subjects", fontsize=16)

colors = plt.cm.tab10(np.linspace(0, 1, len(explanation_function_names)))

for idx, acc_key in enumerate(acc_keys):
    for i, func in enumerate(explanation_function_names):
        color = colors[i]
        mean = np.array(avg_performance[acc_key][func])
        std = np.array(std_performance[acc_key][func])
        
        # Plot line with error bands
        axs[idx].plot(ks, mean, label=func, linewidth=2, color=color)
        axs[idx].fill_between(ks, mean - std, mean + std, alpha=0.2, color=color)
    
    axs[idx].set_title(f"{acc_key.replace('_', ' ').title()}", fontsize=14)
    axs[idx].set_xlabel("k value", fontsize=12)
    axs[idx].set_ylabel("Accuracy", fontsize=12)
    axs[idx].set_ylim([0.45, 1])
    axs[idx].grid(True, linestyle='--', alpha=0.7)
    axs[idx].legend(fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig("average_performance_with_error_bands.png", dpi=300)
plt.show()